# 🇧🇩 বাংলা ভয়েস ক্লোনিং — Bangla Voice Clone
### Powered by Coqui XTTS v2 | Zero-Shot Voice Cloning | Free & Open Source

**Instructions:**
1. Run all cells top to bottom (`Runtime → Run all`)
2. Upload your voice sample when prompted
3. Enter your Bangla text
4. Download the generated audio

> 💡 Enable GPU: `Runtime → Change runtime type → T4 GPU` for 5x faster generation

In [ ]:
# ─────────────────────────────────────────
# CELL 1: Install dependencies
# ─────────────────────────────────────────
print('📦 Installing dependencies...')
import subprocess, sys

packages = [
    'TTS==0.22.0',
    'librosa==0.10.2',
    'soundfile==0.12.1',
    'scipy==1.13.0',
    'flask==3.0.3',
    'flask-cors==4.0.1',
    'pyngrok==7.1.5',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print(f'  ✅ {pkg}')

print('\n🎉 All dependencies installed!')

In [ ]:
# ─────────────────────────────────────────
# CELL 2: Check GPU & system
# ─────────────────────────────────────────
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device.upper()}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  Running on CPU — generation will be slower (~2-5 min)')
    print('   Tip: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─────────────────────────────────────────
# CELL 3: Download XTTS v2 model
# ─────────────────────────────────────────
print('🤖 Loading XTTS v2 model (first run downloads ~1.8GB)...')
print('   This may take 5-10 minutes on first run.\n')

from TTS.api import TTS

use_gpu = torch.cuda.is_available()
tts = TTS(
    model_name='tts_models/multilingual/multi-dataset/xtts_v2',
    progress_bar=True,
    gpu=use_gpu,
)

print('\n✅ XTTS v2 model ready!')
print(f'   Languages supported: {len(tts.languages)} including Bangla (bn)')

In [ ]:
# ─────────────────────────────────────────
# CELL 4: Audio preprocessing utilities
# ─────────────────────────────────────────
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path

def preprocess_audio(input_path, output_path=None, target_sr=22050):
    """
    Preprocess voice sample:
    - Mono conversion
    - Resample to 22050 Hz
    - Trim silence
    - Normalize amplitude
    - Enforce 5-30 second length
    """
    if output_path is None:
        output_path = str(input_path).replace('.mp3', '_clean.wav').replace('.wav', '_clean.wav')
    
    y, sr = librosa.load(input_path, sr=None, mono=True)
    
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    
    y, _ = librosa.effects.trim(y, top_db=25)
    
    peak = np.abs(y).max()
    if peak > 0:
        y = y / peak * 0.7
    
    min_s, max_s = 5 * target_sr, 30 * target_sr
    if len(y) < min_s:
        repeats = int(np.ceil(min_s / len(y)))
        y = np.tile(y, repeats)[:max_s]
    elif len(y) > max_s:
        y = y[:max_s]
    
    sf.write(output_path, y, target_sr, subtype='PCM_16')
    duration = len(y) / target_sr
    print(f'✅ Preprocessed: {duration:.1f}s @ {target_sr}Hz')
    return output_path

print('🔧 Audio utilities ready.')

In [ ]:
# ─────────────────────────────────────────
# CELL 5: Upload voice sample
# ─────────────────────────────────────────
from google.colab import files
import os

print('📤 Upload your voice sample (WAV or MP3, 5-30 seconds recommended):')
uploaded = files.upload()

if not uploaded:
    raise ValueError('❌ No file uploaded!')

raw_filename = list(uploaded.keys())[0]
print(f'\n📁 Uploaded: {raw_filename} ({os.path.getsize(raw_filename) / 1024:.1f} KB)')

print('\n🔧 Preprocessing audio...')
clean_sample = preprocess_audio(raw_filename, 'speaker_clean.wav')
print(f'✅ Sample ready: {clean_sample}')

In [ ]:
# ─────────────────────────────────────────
# CELL 6: Preview uploaded voice
# ─────────────────────────────────────────
from IPython.display import Audio, display
print('🎧 Preview of your voice sample:')
display(Audio(clean_sample))

In [ ]:
# ─────────────────────────────────────────
# CELL 7: Enter Bangla text & generate
# ─────────────────────────────────────────
from IPython.display import Audio, display
import ipywidgets as widgets

# ── Enter your Bangla text here ──
BANGLA_TEXT = "আমি বাংলায় কথা বলতে ভালোবাসি। বাংলাদেশ আমার প্রিয় দেশ।"

print(f'📝 Bangla text: {BANGLA_TEXT}')
print(f'   Length: {len(BANGLA_TEXT)} characters')
print('\n🎙️ Generating cloned Bangla speech...')
print('   (This may take 30 seconds to 5 minutes depending on device)\n')

import time
start = time.time()

output_file = 'bangla_cloned_output.wav'

tts.tts_to_file(
    text=BANGLA_TEXT,
    speaker_wav=clean_sample,
    language='bn',
    file_path=output_file,
)

elapsed = time.time() - start
print(f'\n✅ Done in {elapsed:.1f}s!')
print(f'📁 Output: {output_file}')

print('\n🎧 Cloned voice preview:')
display(Audio(output_file))

In [ ]:
# ─────────────────────────────────────────
# CELL 8: Download output
# ─────────────────────────────────────────
from google.colab import files
print('⬇️  Downloading cloned audio...')
files.download(output_file)
print('✅ Downloaded!')

In [ ]:
# ─────────────────────────────────────────
# CELL 9: Launch Web UI with ngrok tunnel
# (Optional — run full Flask app in Colab)
# ─────────────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok

# Write a minimal Flask app inline
FLASK_APP_CODE = '''
import io, uuid, numpy as np, librosa, soundfile as sf
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from TTS.api import TTS
import torch

app = Flask(__name__)
CORS(app)

use_gpu = torch.cuda.is_available()
tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2", gpu=use_gpu)

@app.route("/health")
def health():
    return {"status": "ok", "model_loaded": True, "language": "bn"}

@app.route("/clone", methods=["POST"])
def clone():
    file = request.files["voice_sample"]
    text = request.form.get("text", "")
    sid = str(uuid.uuid4())[:6]
    raw = f"/tmp/raw_{sid}.wav"
    clean = f"/tmp/clean_{sid}.wav"
    out = f"/tmp/out_{sid}.wav"
    file.save(raw)
    y, sr = librosa.load(raw, sr=None, mono=True)
    y = librosa.resample(y, orig_sr=sr, target_sr=22050)
    y, _ = librosa.effects.trim(y, top_db=25)
    sf.write(clean, y, 22050)
    tts.tts_to_file(text=text, speaker_wav=clean, language="bn", file_path=out)
    return send_file(out, mimetype="audio/wav")

app.run(host="0.0.0.0", port=8000)
'''

with open('/tmp/colab_app.py', 'w') as f:
    f.write(FLASK_APP_CODE)

# Start Flask in background
proc = subprocess.Popen(['python', '/tmp/colab_app.py'])
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f'\n🌐 Web UI available at: {public_url}')
print('   Open this URL and use the frontend HTML files to interact with the API.')
print('   Set API_BASE in app.js to this URL.')

## 🧠 Auto-Learning: Batch Generation

Run this cell to generate multiple sentences and build a Bangla dataset.

In [ ]:
# ─────────────────────────────────────────
# CELL 10: Batch Bangla generation
# Auto-learning dataset builder
# ─────────────────────────────────────────
import json, os
from pathlib import Path
from IPython.display import Audio, display

BANGLA_SENTENCES = [
    "আমি বাংলায় কথা বলতে ভালোবাসি।",
    "বাংলাদেশ আমার প্রিয় দেশ।",
    "আজকের আবহাওয়া অনেক সুন্দর।",
    "কৃত্রিম বুদ্ধিমত্তা ভবিষ্যতের প্রযুক্তি।",
    "বাংলা ভাষা আমার মাতৃভাষা।",
]

dataset_dir = Path('bangla_dataset')
dataset_dir.mkdir(exist_ok=True)

metadata = []

for i, sentence in enumerate(BANGLA_SENTENCES):
    out_file = dataset_dir / f'sample_{i+1:03d}.wav'
    print(f'[{i+1}/{len(BANGLA_SENTENCES)}] Generating: {sentence[:40]}...')
    
    tts.tts_to_file(
        text=sentence,
        speaker_wav=clean_sample,
        language='bn',
        file_path=str(out_file),
    )
    
    metadata.append({'audio': str(out_file), 'text': sentence, 'language': 'bn'})
    print(f'  ✅ Saved: {out_file}')

# Save metadata
with open(dataset_dir / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'\n🎉 Dataset built: {len(metadata)} samples in ./{dataset_dir}/')

# Preview last generated
print('\n🎧 Preview (last generated):')
display(Audio(str(out_file)))